![image.png](https://i.imgur.com/a3uAqnb.png)


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammedtawfikmusaed/asthma-detection-dataset-version-2")

print("Path to dataset files:", path)

In [ ]:
# =========================================================
# Imports & global config
# =========================================================
import os, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchaudio
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Paths / classes
DATA_DIR = Path("/kaggle/input/asthma-detection-dataset-version-2/Asthma Detection Dataset Version 2/Asthma Detection Dataset Version 2")
CLASS_NAMES = ["asthma", "Bronchial", "copd", "healthy", "pneumonia"]
label2id = {c:i for i,c in enumerate(CLASS_NAMES)}
id2label = {i:c for c,i in label2id.items()}

# Audio params
TARGET_SR = 4000       # widely used for lung sounds
CLIP_SECONDS = 5.0     # dataset clips ~1.5–5s → fix to 5s
MAX_LEN = int(TARGET_SR * CLIP_SECONDS)

# Training
BATCH_SIZE = 64
EPOCHS = 35
LR = 2e-3
WD = 1e-4
NUM_FOLDS = 5
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = torch.cuda.is_available()


In [ ]:
# =========================================================
# Build dataframe from directory structure
# =========================================================
def build_df(root: Path):
    rows = []
    # accept case variants like "Bronchial"
    present = {p.name.lower(): p for p in root.iterdir() if p.is_dir()}
    for cls in CLASS_NAMES:
        sub = present.get(cls.lower(), None)
        if sub is None:
            # try capitalized just in case
            sub = root / cls.capitalize()
        if not sub.exists():
            print(f"[WARN] Missing class folder: {cls} -> {sub}")
            continue
        for wav in sub.rglob("*.wav"):
            rows.append({"path": str(wav), "label": label2id[cls]})
    df = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print(df.label.value_counts().sort_index(), " (by label id)")
    return df

df = build_df(DATA_DIR)
print("Total files:", len(df))


In [ ]:
# =========================================================
# Audio loading utils
# =========================================================
def load_wave_fixed(path):
    wav, sr = torchaudio.load(path)             # [C, T]
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)         # mono [1, T]
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    x = wav[0]                                   # [T]
    # simple amplitude normalization (robust)
    x = x / (x.abs().max() + 1e-8)
    # pad / crop to fixed length
    if x.numel() < MAX_LEN:
        x = F.pad(x, (0, MAX_LEN - x.numel()))
    else:
        x = x[:MAX_LEN]
    return x.unsqueeze(0)                        # [1, T]


In [ ]:
# =========================================================
# Dataset & per-fold dataloaders
# =========================================================
class LungSoundsDataset(Dataset):
    def __init__(self, dframe: pd.DataFrame):
        self.df = dframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = load_wave_fixed(row["path"])        # [1, T]
        y = int(row["label"])
        return x, y

def make_loaders(train_df, val_df):
    train_ds = LungSoundsDataset(train_df)
    val_ds   = LungSoundsDataset(val_df)
    tr = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return tr, va

# Class weights for imbalance (inverse freq)
counts = df["label"].value_counts().sort_index().values.astype(np.float32)
class_weights = torch.tensor(counts.sum() / (counts + 1e-6), dtype=torch.float32)
class_weights = (class_weights / class_weights.mean()).to(DEVICE)
print("Class weights:", class_weights.cpu().numpy())


In [ ]:
# =========================================================
# WaveNet (causal dilated convs) for raw audio classification
# =========================================================
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm

class CausalConv1d(nn.Module):
    """1D causal conv implemented via left padding + no internal padding."""
    def __init__(self, in_ch, out_ch, kernel_size=2, dilation=1):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                                          padding=0, dilation=dilation, bias=True)
    def forward(self, x):
        x = F.pad(x, (self.pad, 0))  # left-pad only → causal
        return self.conv(x)

class WaveNetResidualBlock(nn.Module):
    """
    Gated residual unit:
      z = tanh(Conv_dil(x)) * sigmoid(Conv_dil(x))
      residual = Conv1x1(z) + x
      skip     = Conv1x1(z)
    """
    def __init__(self, channels, skip_channels, kernel_size=2, dilation=1, dropout=0.05):
        super().__init__()
        self.filter = CausalConv1d(channels, channels, kernel_size, dilation)
        self.gate   = CausalConv1d(channels, channels, kernel_size, dilation)
        self.dropout = nn.Dropout(dropout)
        self.res_conv  = nn.Conv1d(channels, channels, kernel_size=1)
        self.skip_conv = nn.Conv1d(channels, skip_channels, kernel_size=1)

    def forward(self, x):
        f = torch.tanh(self.filter(x))
        g = torch.sigmoid(self.gate(x))
        z = self.dropout(f * g)
        res = self.res_conv(z)
        skip = self.skip_conv(z)
        # scale residual sum a bit to keep activations stable in deep stacks
        out = x + res
        return out, skip

class WaveNetClassifier(nn.Module):
    """
    Stacked dilated causal conv blocks with skip aggregation and global pooling.
    Suitable for clip-level classification on raw waveforms [B, 1, T].
    """
    def __init__(self, num_classes, in_channels=1, res_channels=64, skip_channels=128,
                 kernel_size=2, layers_per_stack=10, num_stacks=2, dropout=0.05):
        super().__init__()
        self.input_proj = nn.Conv1d(in_channels, res_channels, kernel_size=1)

        blocks = []
        for _ in range(num_stacks):
            for l in range(layers_per_stack):
                dilation = 2 ** l                     # 1, 2, 4, ..., 2^(L-1)
                blocks.append(WaveNetResidualBlock(
                    channels=res_channels,
                    skip_channels=skip_channels,
                    kernel_size=kernel_size,
                    dilation=dilation,
                    dropout=dropout
                ))
        self.blocks = nn.ModuleList(blocks)

        # Post-processing of aggregated skip connections
        self.out = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(skip_channels, skip_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(skip_channels, num_classes, kernel_size=1),
        )

    def forward(self, x):                      # x: [B, 1, T]
        x = self.input_proj(x)                 # [B, C, T]
        skip_sum = None
        for block in self.blocks:
            x, skip = block(x)                 # residual path + skip output
            skip_sum = skip if skip_sum is None else (skip_sum + skip)
        y = self.out(F.relu(skip_sum))         # [B, num_classes, T]
        y = y.mean(dim=-1)                     # global average over time
        return y                               # logits [B, num_classes]

model = WaveNetClassifier(num_classes=len(CLASS_NAMES)).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M params")


In [ ]:
# =========================================================
# Optimizer, scheduler, loss; eval helpers
# =========================================================
def make_optimizer(model):
    return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

def train_one_epoch(model, loader, criterion, optimizer, scaler=None):
    model.train()
    losses = []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_tgts = [], []
    losses = []
    criterion_eval = nn.CrossEntropyLoss()
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = criterion_eval(logits, y)
        preds = logits.argmax(1)
        losses.append(loss.item())
        all_preds.append(preds.detach().cpu().numpy())
        all_tgts.append(y.detach().cpu().numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_tgts)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    return float(np.mean(losses)), acc, f1m, y_true, y_pred


In [ ]:
# =========================================================
# Train across folds (set RUN_SINGLE_FOLD to train one fold fast)
# =========================================================
RUN_SINGLE_FOLD = True
SINGLE_FOLD_IDX = 0

skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(df["path"], df["label"]))

selected_folds = [folds[SINGLE_FOLD_IDX]] if RUN_SINGLE_FOLD else folds

history = []
train_losses = []
val_losses   = []

for fold_idx, (tr_idx, va_idx) in enumerate(selected_folds):
    print(f"\n========== Fold {fold_idx} ==========")
    tr_df, va_df = df.iloc[tr_idx], df.iloc[va_idx]
    tr_loader, va_loader = make_loaders(tr_df, va_df)

    model = AudioCNN1D(num_classes=len(CLASS_NAMES)).to(DEVICE)
    optimizer = make_optimizer(model)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_f1 = -1.0
    patience, patience_limit = 0, 6
    ckpt_path = f"best_fold{fold_idx}.pt"

    for epoch in range(1, EPOCHS+1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, tr_loader, criterion, optimizer, scaler)
        va_loss, va_acc, va_f1, y_true, y_pred = evaluate(model, va_loader)
        train_losses.append(tr_loss)
        val_losses.append(va_loss)
        dt = time.time() - t0
        print(f"Epoch {epoch:02d} | tr_loss {tr_loss:.4f} | va_loss {va_loss:.4f} | "
              f"acc {va_acc:.4f} | f1_macro {va_f1:.4f} | {dt:.1f}s")

        if va_f1 > best_f1:
            best_f1 = va_f1
            patience = 0
            torch.save({"model": model.state_dict(),
                        "cfg": {"sr": TARGET_SR, "len": MAX_LEN, "classes": CLASS_NAMES}}, ckpt_path)
        else:
            patience += 1
            if patience >= patience_limit:
                print("Early stopping.")
                break

    # Load best & final eval
    state = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state["model"])
    va_loss, va_acc, va_f1, y_true, y_pred = evaluate(model, va_loader)
    print("\nBest checkpoint results:")
    print(f"VAL acc={va_acc:.4f}  macroF1={va_f1:.4f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
    history.append({"fold": fold_idx, "acc": va_acc, "f1_macro": va_f1})


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, 'b-o', label='Training Loss')
plt.plot(epochs, val_losses,   'r-o', label='Validation Loss')
plt.title('Training & Validation Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()
